# 03 — Full corpus → MySQL (chunked Batch API)

**Run All is supported.** After each chunk succeeds, results are downloaded from the Batch result **file** (not inline) and upserted into `message_sentiment`.

If a run already finished on Google but DB is empty/partial, use:
` .venv/bin/python scripts/recover_batch_results.py `

Set `CONFIRM_FULL_RUN = True` before Run All.

In [ ]:
from pathlib import Path
import sys
import importlib

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import sentiment.client as _client_mod
import sentiment.store as _store_mod
import sentiment.batch_results as _br_mod
importlib.reload(_client_mod)
importlib.reload(_store_mod)
importlib.reload(_br_mod)

from sentiment.batch_results import parse_batch_job
from sentiment.client import make_client
from sentiment.config import ensure_data_dirs, get_pipeline_config
from sentiment.estimate import estimate_from_windows_df
from sentiment.io import append_checkpoint, checkpoint_path, finalize_results, load_scored_ids
from sentiment.schema import results_to_dataframe
from sentiment.store import (
    count_sentiment_rows,
    ensure_sentiment_table,
    filter_windows_not_in_db,
    upsert_sentiment_dataframe,
)
from sentiment.windows import load_windows_parquet

paths = ensure_data_dirs()
cfg = get_pipeline_config(require_mysql=True)
RUN_NAME = "full_corpus"
LINES_PER_CHUNK = 2500
POLL_SECONDS = 60
CONFIRM_FULL_RUN = False  # set True to submit new chunks

ensure_sentiment_table(cfg.mysql)
print("Model:", cfg.model, flush=True)
print("DB rows already:", f"{count_sentiment_rows(cfg.mysql):,}", flush=True)

In [ ]:
windows_all = load_windows_parquet()
windows = filter_windows_not_in_db(cfg.mysql, windows_all)
local_done = load_scored_ids(RUN_NAME)
if local_done:
    windows = windows[~windows["message_id"].astype(str).isin(local_done)].reset_index(drop=True)

est = estimate_from_windows_df(windows, batch_size=cfg.batch_size)
n_chunks = (
    max(1, (est.n_batches + LINES_PER_CHUNK - 1) // LINES_PER_CHUNK)
    if est.n_batches
    else 0
)
print(f"All windows:     {len(windows_all):,}", flush=True)
print(f"Still to score:  {len(windows):,}", flush=True)
print(f"Chunk jobs:      ~{n_chunks}", flush=True)
print(est.as_dict(), flush=True)

In [ ]:
def on_chunk_done(job, chunk_i, chunk_n):
    client_local = make_client()
    results = parse_batch_job(client_local.client, job)
    if not results:
        print(f"  WARN: no parsable results for chunk {chunk_i}", flush=True)
        return
    append_checkpoint(RUN_NAME, results)
    n = upsert_sentiment_dataframe(
        cfg.mysql, results_to_dataframe(results), model=cfg.model
    )
    print(
        f"  Upserted {n:,} from chunk {chunk_i}/{chunk_n} | "
        f"DB now {count_sentiment_rows(cfg.mysql):,}",
        flush=True,
    )


if not CONFIRM_FULL_RUN:
    print("CONFIRM_FULL_RUN is False — not submitting.", flush=True)
elif len(windows) == 0:
    print("Nothing left to score.", flush=True)
else:
    client = make_client()
    chunk_paths = client.write_batch_jsonl_chunks(
        windows,
        paths["results"] / "batch_chunks",
        batch_size=cfg.batch_size,
        lines_per_chunk=LINES_PER_CHUNK,
        prefix="full",
    )
    client.run_chunked_batch(
        chunk_paths,
        on_job_succeeded=on_chunk_done,
        poll_seconds=POLL_SECONDS,
        display_prefix="discord-sentiment",
    )
    print("DB total:", f"{count_sentiment_rows(cfg.mysql):,}", flush=True)

In [ ]:
# Optional: recover already-paid Batch files without re-scoring
RUN_RECOVER = False
if RUN_RECOVER:
    import runpy
    runpy.run_path(str(ROOT / "scripts" / "recover_batch_results.py"), run_name="__main__")
else:
    print("Set RUN_RECOVER=True to re-download prior Batch result files.", flush=True)

In [ ]:
from sentiment.store import fetch_scored_message_ids

ckpt = checkpoint_path(RUN_NAME)
if ckpt.exists():
    out = finalize_results(RUN_NAME, output_name="sentiment_results")
    results = pd.read_parquet(out)
    print(f"Parquet: {len(results):,} → {out}", flush=True)

db_n = count_sentiment_rows(cfg.mysql)
windows_all = load_windows_parquet()
missing = set(windows_all["message_id"].astype(str)) - fetch_scored_message_ids(cfg.mysql)
print(f"DB total: {db_n:,}", flush=True)
print(f"Missing vs windows: {len(missing):,} / {len(windows_all):,}", flush=True)